# COM6003 CW1 — Buckinghamshire EPC Analysis

**Author:** Harry Lloyd (22224922)
**Module:** COM6003 Data Science
**Dataset:** Domestic Energy Performance Certificates, Buckinghamshire (`E06000060`)
**Source:** [EPC Open Data](https://epc.opendatacommunities.org/) — Ministry of Housing, Communities and Local Government

This notebook performs the full analysis pipeline:

1. Data acquisition & understanding
2. Data wrangling (cleaning, deduplication, outlier handling)
3. Feature engineering
4. Descriptive analytics
5. Diagnostic analytics
6. Predictive modelling
7. Feature importance and recommendations

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

DATA_RAW = '../data/raw/certificates.csv'
DATA_CLEAN = '../data/processed/bucks_epc_clean.csv'
FIG_DIR = '../figures/'

RANDOM_STATE = 42

## 1. Data Acquisition & Understanding

The dataset was downloaded from the EPC Open Data portal on 22 May 2026,
filtered to the Buckinghamshire local authority (`E06000060`). The portal
is the official government source for all Energy Performance Certificates
issued under the Energy Performance of Buildings (England and Wales)
Regulations 2012.

**Collection methodology:** Each EPC is produced by an accredited Domestic
Energy Assessor (DEA) using the Reduced Standard Assessment Procedure
(RdSAP), a simplified version of SAP (BRE, 2014). The assessor inspects
the property and records standardised observations about the fabric,
heating system, and lighting, which feed into a deterministic calculation
of the SAP score (0–100) and a corresponding letter band (A–G).

**Important biases and limitations:**

- **Selection bias:** EPCs are only required at point of sale, let, or
  major refurbishment. Properties that rarely change hands are
  systematically under-represented.
- **Assessor variability:** Independent studies (Crawley et al., 2019)
  have shown meaningful between-assessor variation for the same property.
- **Model vs measured:** RdSAP estimates *theoretical* energy demand from
  a standard occupancy assumption, not actual measured consumption.
  Real bills can diverge substantially.
- **Default assumptions:** Where data is missing during inspection, RdSAP
  uses construction-age-band defaults — which can mask true variation.

We acknowledge these limitations explicitly in the report's Conclusion.

In [2]:
df = pd.read_csv(DATA_RAW, low_memory=False)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB')

Shape: 204,347 rows × 93 columns


Memory: 770 MB


In [3]:
df.head(3)

,certificate_number,address1,address2,address3,address,postcode,inspection_date,uprn,environment_impact_potential,energy_consumption_current,energy_consumption_potential,environment_impact_current,co2_emissions_current,co2_emiss_curr_per_floor_area,co2_emissions_potential,total_floor_area,lodgement_date,report_type,posttown,lodgement_datetime,current_energy_efficiency,current_energy_rating,potential_energy_efficiency,potential_energy_rating,extension_count,number_open_fireplaces,number_heated_rooms,number_habitable_rooms,low_energy_lighting,low_energy_fixed_lighting_outlets_count,solar_water_heating_flag,mechanical_ventilation,tenure,property_type,transaction_type,construction_age_band,built_form,energy_tariff,glazed_type,glazed_area,heat_loss_corridor,main_fuel,unheated_corridor_length,floor_level,flat_top_storey,flat_storey_count,mains_gas_flag,photo_supply,wind_turbine_count,lighting_cost_current,lighting_cost_potential,heating_cost_current,heating_cost_potential,hot_water_cost_current,hot_water_cost_potential,multi_glaze_proportion,hotwater_description,hot_water_energy_eff,hot_water_env_eff,floor_description,floor_energy_eff,floor_env_eff,roof_description,roof_energy_eff,roof_env_eff,walls_description,walls_energy_eff,walls_env_eff,windows_description,windows_energy_eff,windows_env_eff,secondheat_description,sheating_energy_eff,sheating_env_eff,mainheat_description,mainheat_energy_eff,mainheat_env_eff,mainheatcont_description,mainheatc_energy_eff,mainheatc_env_eff,lighting_description,lighting_energy_eff,lighting_env_eff,fixed_lighting_outlets_count,floor_height,main_heating_controls,local_authority,local_authority_label,constituency_label,constituency,country,region,uprn_source
0,2315-3061-1204-7816-8204,12 Westmorland Avenue,NaN,NaN,12 Westmorland Avenue,HP21 7HW,2026-04-29,7.660190e+08,80,171,107,69,3.2,30.0,2.1,106,2026-04-29,2,AYLESBURY,2026-04-29 14:31:08,69,C,80,C,0.0,NaN,6.0,6.0,100.0,22.0,N,natural,owner-occupied,House,Marketed sale,England and Wales: 1950-1966,Semi-Detached,Single,NaN,NaN,NaN,mains gas (not community),NaN,NaN,N,2.0,Y,0.0,0.0,62,62,1032,697,374,354,100.0,From main system,Good,Good,"Solid, no insulation (assumed)",NaN,NaN,"Pitched, 175 mm loft insulation",Good,Good,"Cavity wall, as built, no insulation (assumed)",Poor,Poor,Fully double glazed,Average,Average,NaN,NaN,NaN,"Boiler and radiators, mains gas",Good,Good,"Programmer, room thermostat and TRVs",Good,Good,Good lighting efficiency,Good,Good,22.0,2.30,"Programmer, room thermostat and TRVs",E06000060,Buckinghamshire,Aylesbury,E14001071,England,E12000008,Energy Assessor
1,2347-3061-3204-2346-8204,33 The Warren,NaN,NaN,33 The Warren,HP5 2RX,2026-04-24,1.000805e+11,69,189,159,65,4.9,35.0,4.2,141,2026-04-29,2,CHESHAM,2026-04-29 19:25:17,69,C,75,C,2.0,NaN,7.0,7.0,100.0,25.0,N,natural,owner-occupied,House,Marketed sale,England and Wales: 1967-1975,Detached,Single,NaN,NaN,NaN,mains gas (not community),NaN,NaN,N,2.0,Y,0.0,0.0,77,77,1571,1378,269,270,100.0,From main system,Good,Good,"Solid, no insulation (assumed)",NaN,NaN,"Pitched, 100 mm loft insulation",Average,Average,"Cavity wall, filled cavity",Good,Good,Fully double glazed,Average,Average,NaN,NaN,NaN,"Boiler and radiators, mains gas",Good,Good,"Programmer, room thermostat and TRVs",Good,Good,Good lighting efficiency,Good,Good,25.0,2.43,"Programmer, room thermostat and TRVs",E06000060,Buckinghamshire,Mid Buckinghamshire,E14001360,England,E12000008,Energy Assessor
2,0100-8653-0922-5624-3463,The Stable,Little Kings Ash Farm,Chesham Lane,"The Stable, Little Kings Ash Farm, Chesham Lane",HP16 9NP,2026-04-25,1.001378e+10,50,415,392,49,3.0,87.0,2.9,35,2026-04-29,2,KINGS ASH,2026-04-29 19:31:10,59,D,64,D,0.0,NaN,2.0,2.0,100.0,12.0,N,natural,owner-occupied,Bungalow,Marketed sale,England and Wales: 2007-2011,End-Terrace,Single,NaN,NaN,NaN,LPG (community),NaN,NaN,N,1.0,N,0.0,0.0,24,24,613,613,216,216,100.0,Community scheme,Average,Poor,"Solid, insulated (assumed)",NaN,NaN,"Pitched, insulated",Good,Good,"Cavi

In [4]:
# Inspection date range — how recent and how broad is the data?
df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
print(f'Earliest inspection: {df["inspection_date"].min().date()}')
print(f'Latest inspection:   {df["inspection_date"].max().date()}')

Earliest inspection: 2011-02-11
Latest inspection:   2026-04-30


In [5]:
# Target variable distribution
target_dist = df['current_energy_rating'].value_counts().sort_index()
print('Current energy rating distribution:')
print(target_dist)
print(f'\nTotal: {target_dist.sum():,}')

Current energy rating distribution:
current_energy_rating
A     1019
B    29442
C    71395
D    70458
E    24744
F     5925
G     1364
Name: count, dtype: int64

Total: 204,347


## 2. Data Wrangling

The raw data has several quality issues that need to be addressed before
analysis:

1. **Duplicates** — the same property (UPRN) may have multiple EPCs over
   time as it gets re-inspected. We keep only the most recent certificate
   per property.
2. **Empty columns** — some columns are 100% missing (e.g.
   `sheating_energy_eff`) and provide no information.
3. **Categorical inconsistencies** — `tenure`, `main_fuel`, and
   `construction_age_band` have multiple labels referring to the same
   category due to historical schema changes. These need to be harmonised.
4. **Outliers** — some numeric columns contain implausible values
   (`current_energy_efficiency` > 100, `total_floor_area` > 1,000 m²).
5. **Missing values** — handled column by column based on the nature of
   the variable and its missingness pattern.

### 2.1 Deduplicate — keep latest certificate per property

In [6]:
# Some UPRNs have many certificates. Keep only the most recent for each property.
print(f'Before dedup:  {len(df):,} rows')
print(f'Unique UPRNs:  {df["uprn"].nunique():,}')
print(f'Missing UPRNs: {df["uprn"].isna().sum():,}')

# For rows without UPRN, fall back to certificate_number (always unique)
df['_uprn_key'] = df['uprn'].fillna(df['certificate_number'])

df = (df
      .sort_values('inspection_date')
      .drop_duplicates(subset='_uprn_key', keep='last')
      .drop(columns='_uprn_key')
      .reset_index(drop=True))

print(f'After dedup:   {len(df):,} rows')

Before dedup:  204,347 rows
Unique UPRNs:  156,021
Missing UPRNs: 1,296


After dedup:   157,317 rows


### 2.2 Drop fully-empty and irrelevant columns

In [7]:
# Columns that are 100% missing carry no information
all_null = df.columns[df.isna().all()].tolist()
print(f'Fully-null columns to drop: {all_null}')

# Free-text address fields and IDs are not predictive
admin_cols = ['certificate_number', 'address1', 'address2', 'address3',
              'address', 'uprn', 'lodgement_date', 'lodgement_datetime',
              'report_type', 'uprn_source', 'country', 'region',
              'local_authority', 'local_authority_label',
              'constituency', 'constituency_label']

# Columns with >85% missing rarely add predictive signal and risk overfitting
high_missing = df.columns[df.isna().mean() > 0.85].tolist()
print(f'High-missing (>85%) columns to drop: {high_missing}')

to_drop = list(set(all_null + admin_cols + high_missing))
df = df.drop(columns=[c for c in to_drop if c in df.columns])
print(f'\nColumns remaining: {df.shape[1]}')

Fully-null columns to drop: ['sheating_energy_eff', 'sheating_env_eff']


High-missing (>85%) columns to drop: ['address3', 'unheated_corridor_length', 'floor_energy_eff', 'floor_env_eff', 'sheating_energy_eff', 'sheating_env_eff']

Columns remaining: 72


### 2.3 Harmonise categorical inconsistencies

In [8]:
# Tenure has casing inconsistencies — 'owner-occupied' and 'Owner-occupied' are the same
df['tenure'] = df['tenure'].str.lower().str.strip()
print('Tenure after lowercasing:')
print(df['tenure'].value_counts(dropna=False))

Tenure after lowercasing:
tenure
owner-occupied      87634
rented (social)     23422
unknown             22709
rented (private)    18742
NaN                  4810
Name: count, dtype: int64


In [9]:
# main_fuel has multiple labels referring to the same fuel + some \n artefacts
df['main_fuel'] = df['main_fuel'].str.replace('\n', '', regex=False).str.strip().str.lower()

def simplify_fuel(s):
    if pd.isna(s):
        return np.nan
    s = s.lower()
    if 'mains gas' in s or s.startswith('gas: mains'):
        return 'mains gas'
    if 'lpg' in s:
        return 'lpg'
    if 'electricity' in s:
        return 'electricity'
    if 'oil' in s:
        return 'oil'
    if 'wood' in s or 'biomass' in s or 'logs' in s or 'pellets' in s or 'chips' in s:
        return 'biomass'
    if 'coal' in s or 'anthracite' in s or 'smokeless' in s:
        return 'solid fuel'
    if 'community' in s:
        return 'community heating'
    if 'dual fuel' in s:
        return 'dual fuel'
    return 'other'

df['main_fuel_grouped'] = df['main_fuel'].apply(simplify_fuel)
print(df['main_fuel_grouped'].value_counts(dropna=False))

main_fuel_grouped
mains gas            126840
electricity           19678
oil                    7165
lpg                    1854
NaN                    1181
biomass                 291
community heating       196
solid fuel               83
other                    29
Name: count, dtype: int64


In [10]:
# construction_age_band has two formats: 'England and Wales: 1950-1966' vs just '2019'
def parse_age_band(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace('England and Wales:', '').strip()
    if 'before 1900' in s.lower():
        return 1899
    if 'onwards' in s.lower():
        # e.g. '2012 onwards' — take the start year
        return int(s.split()[0])
    if '-' in s:
        # range like '1950-1966' — take midpoint
        try:
            a, b = s.split('-')
            return (int(a.strip()) + int(b.strip())) / 2
        except ValueError:
            return np.nan
    # single year like '2019'
    try:
        return int(s)
    except ValueError:
        return np.nan

df['construction_year'] = df['construction_age_band'].apply(parse_age_band)
print(df['construction_year'].describe())

count    153884.000000
mean       1973.871182
std          34.450899
min        1780.000000
25%        1958.000000
50%        1971.000000
75%        2004.500000
max        3013.000000
Name: construction_year, dtype: float64


### 2.4 Outlier handling

In [11]:
# current_energy_efficiency should be 0-100 by definition (SAP scale)
print('Before:', df['current_energy_efficiency'].describe()[['min', 'max']].to_dict())
bad = ((df['current_energy_efficiency'] < 1) | (df['current_energy_efficiency'] > 100)).sum()
print(f'Implausible SAP scores: {bad}')
df = df[(df['current_energy_efficiency'] >= 1) & (df['current_energy_efficiency'] <= 100)].copy()
print('After: ', df['current_energy_efficiency'].describe()[['min', 'max']].to_dict())

Before: {'min': 1.0, 'max': 277.0}
Implausible SAP scores: 85


After:  {'min': 1.0, 'max': 100.0}


In [12]:
# total_floor_area — anything above the 99.5th percentile is implausible for a domestic dwelling
fa_cap = df['total_floor_area'].quantile(0.995)
print(f'99.5th percentile floor area: {fa_cap:.1f} m²')
print(f'Records above this: {(df["total_floor_area"] > fa_cap).sum()}')

# Cap rather than drop (preserve sample size, follow standard winsorisation)
df['total_floor_area'] = df['total_floor_area'].clip(upper=fa_cap)

# Also drop properties with implausibly small areas (<10 m²)
small = (df['total_floor_area'] < 10).sum()
print(f'Records with <10 m² floor area (dropped): {small}')
df = df[df['total_floor_area'] >= 10].copy()

99.5th percentile floor area: 481.8 m²
Records above this: 787
Records with <10 m² floor area (dropped): 24


### 2.5 Missing-value strategy

In [13]:
# Summary of remaining missingness
miss = df.isna().sum().sort_values(ascending=False)
miss_pct = (miss / len(df) * 100).round(1)
remaining_missing = pd.DataFrame({'n_missing': miss, 'pct_missing': miss_pct})
remaining_missing = remaining_missing[remaining_missing['n_missing'] > 0]
remaining_missing.head(25)

,n_missing,pct_missing
heat_loss_corridor,131380,83.6
floor_level,118291,75.2
secondheat_description,103197,65.6
glazed_area,56094,35.7
glazed_type,56094,35.7
photo_supply,46998,29.9
mains_gas_flag,45396,28.9
mechanical_ventilation,27840,17.7
extension_count,27839,17.7
solar_water_heating_flag,27839,17.7


In [14]:
# Numeric: impute with median (robust to outliers)
# Categorical: impute with 'unknown' (preserve the missingness as a category)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Don't impute the target!
TARGETS = ['current_energy_efficiency', 'current_energy_rating']
numeric_cols = [c for c in numeric_cols if c not in TARGETS]

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna('unknown')

print(f'Remaining missing: {df.isna().sum().sum()}')
print(f'Final shape: {df.shape}')

Remaining missing: 0
Final shape: (157208, 74)


### 2.6 Save cleaned dataset

In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save uncompressed for fast loading in subsequent cells
df.to_csv(DATA_CLEAN, index=False)

# Also save a gzipped copy for the repo (the uncompressed file is >100MB,
# which exceeds GitHub's per-file limit)
df.to_csv(DATA_CLEAN + '.gz', index=False, compression='gzip')

print(f'Saved: {DATA_CLEAN}  ({os.path.getsize(DATA_CLEAN) / 1e6:.1f} MB)')
print(f'Saved: {DATA_CLEAN}.gz  ({os.path.getsize(DATA_CLEAN + ".gz") / 1e6:.1f} MB)')
print(f'Shape: {df.shape}')

Saved: ../data/processed/bucks_epc_clean.csv  (116.5 MB)
Saved: ../data/processed/bucks_epc_clean.csv.gz  (11.5 MB)
Shape: (157208, 74)


## 3. Feature Engineering

The cleaned dataset gives us 73 columns of raw observations. To extract
predictive signal, we engineer a smaller set of more informative features:

1. **Ordinal encoding** of the seven `*_energy_eff` columns
   (`Very Poor` → `Very Good` as 1–5). These are inherently ordered.
2. **Composite scores** — a *fabric efficiency* score (envelope: walls,
   roof, windows, floor) and a *systems efficiency* score (heating,
   hot water, lighting). Composite scores capture related-feature
   signal that may be noisy at the individual level.
3. **Building age** — derived from the parsed `construction_year`.
4. **Energy intensity** — `energy_consumption_current / total_floor_area`
   (kWh/m²/yr). This normalises for property size and is the metric
   used in most academic EPC literature (Hardy & Glew, 2019).
5. **Log-transformed floor area** — `total_floor_area` is right-skewed;
   the log transform improves linearity for regression models.
6. **Heating cost intensity** — `heating_cost_current / total_floor_area`.
7. **Simplified categorical groupings** for `built_form` (rare categories
   merged) and a derived `postcode_district` for geographic signal.

### 3.1 Load the cleaned data

In [16]:
# Reload the cleaned dataset (so this section can be re-run independently)
df = pd.read_csv(DATA_CLEAN, low_memory=False)
df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
print(f'Shape: {df.shape}')

Shape: (157208, 74)


### 3.2 Ordinal encoding of efficiency ratings

In [17]:
# All *_energy_eff columns use the same 5-point ordinal scale
EFF_MAP = {
    'Very Poor': 1,
    'Poor': 2,
    'Average': 3,
    'Good': 4,
    'Very Good': 5,
    'unknown': np.nan,
    'N/A': np.nan,
}

eff_cols = [c for c in df.columns if c.endswith('_energy_eff')]
print(f'Ordinal-encoding {len(eff_cols)} columns: {eff_cols}')

for c in eff_cols:
    df[c + '_ord'] = df[c].map(EFF_MAP)

# Quick check
df[[c + '_ord' for c in eff_cols]].describe()

Ordinal-encoding 7 columns: ['hot_water_energy_eff', 'roof_energy_eff', 'walls_energy_eff', 'windows_energy_eff', 'mainheat_energy_eff', 'mainheatc_energy_eff', 'lighting_energy_eff']


,hot_water_energy_eff_ord,roof_energy_eff_ord,walls_energy_eff_ord,windows_energy_eff_ord,mainheat_energy_eff_ord,mainheatc_energy_eff_ord,lighting_energy_eff_ord
count,157207.000000,136409.000000,157206.000000,157206.000000,157207.000000,157206.000000,157207.000000
mean,3.588549,3.581157,3.347442,3.524611,3.709994,3.815446,4.218190
std,0.874616,1.168642,1.265712,0.912620,0.774415,0.789383,1.229953
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,2.000000,3.000000,4.000000,3.000000,4.000000
50%,4.000000,4.000000,4.000000,3.000000,4.000000,4.000000,5.000000
75%,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


### 3.3 Composite fabric and systems scores

In [18]:
# Fabric (passive envelope) — what stops heat escaping
fabric_cols = ['walls_energy_eff_ord', 'roof_energy_eff_ord',
               'windows_energy_eff_ord']
df['fabric_score'] = df[fabric_cols].mean(axis=1)

# Systems (active) — heating, hot water, lighting
system_cols = ['mainheat_energy_eff_ord', 'hot_water_energy_eff_ord',
               'lighting_energy_eff_ord']
df['systems_score'] = df[system_cols].mean(axis=1)

print('Fabric score:')
print(df['fabric_score'].describe())
print()
print('Systems score:')
print(df['systems_score'].describe())

Fabric score:
count    157207.000000
mean          3.491607
std           0.905590
min           1.000000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: fabric_score, dtype: float64

Systems score:
count    157207.000000
mean          3.838911
std           0.664025
min           1.000000
25%           3.333333
50%           4.000000
75%           4.333333
max           5.000000
Name: systems_score, dtype: float64


### 3.4 Building age

In [19]:
# Current year minus parsed construction year. The dataset spans 2008-2026
# inspections so we use the inspection year as the reference, not 'today'.
df['inspection_year'] = df['inspection_date'].dt.year
df['building_age_years'] = df['inspection_year'] - df['construction_year']

# A handful of negative ages (inspection date before recorded construction year)
neg = (df['building_age_years'] < 0).sum()
print(f'Negative ages (data error, clipped to 0): {neg}')
df['building_age_years'] = df['building_age_years'].clip(lower=0, upper=200)

print(df['building_age_years'].describe())

Negative ages (data error, clipped to 0): 510


count    157208.000000
mean         45.882356
std          34.087382
min           0.000000
25%          16.000000
50%          45.000000
75%          64.000000
max         200.000000
Name: building_age_years, dtype: float64


### 3.5 Energy and cost intensity (per m²)

In [20]:
# Energy consumption per square metre — the standard EPC intensity metric
df['energy_per_m2'] = df['energy_consumption_current'] / df['total_floor_area']

# Heating cost per square metre (annual £/m²)
df['heating_cost_per_m2'] = df['heating_cost_current'] / df['total_floor_area']

# Log-transformed floor area (right-skewed in raw form)
df['log_floor_area'] = np.log1p(df['total_floor_area'])

# Defensive: replace any infinities from divide-by-zero with NaN
for c in ['energy_per_m2', 'heating_cost_per_m2']:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)
    df[c] = df[c].fillna(df[c].median())

print(df[['energy_per_m2', 'heating_cost_per_m2', 'log_floor_area']].describe())

       energy_per_m2  heating_cost_per_m2  log_floor_area
count  157208.000000        157208.000000   157208.000000
mean        2.923783             7.918362        4.506368
std         3.202717             4.853977        0.517522
min        -2.066667            -0.012987        2.397895
25%         1.263158             4.914286        4.174387
50%         2.241379             7.000000        4.454347
75%         3.543895             9.753127        4.804021
max       368.681818            97.961538        6.179696


### 3.6 Categorical simplification and geographic features

In [21]:
# built_form has rare categories that won't help the model
form_counts = df['built_form'].value_counts()
print('built_form counts:')
print(form_counts)

# Keep the top categories; merge the rest into 'Other'
common_forms = form_counts[form_counts >= 1000].index
df['built_form_simple'] = df['built_form'].where(
    df['built_form'].isin(common_forms), 'Other'
)
print()
print('After simplification:')
print(df['built_form_simple'].value_counts())

built_form counts:
built_form
Detached                50580
Semi-Detached           50191
Mid-Terrace             28980
End-Terrace             20180
Enclosed End-Terrace     2499
Not Recorded             2404
unknown                  1335
Enclosed Mid-Terrace     1039
Name: count, dtype: int64

After simplification:
built_form_simple
Detached                50580
Semi-Detached           50191
Mid-Terrace             28980
End-Terrace             20180
Enclosed End-Terrace     2499
Not Recorded             2404
unknown                  1335
Enclosed Mid-Terrace     1039
Name: count, dtype: int64


In [22]:
# Postcode district — the part before the space (e.g. 'HP11' from 'HP11 2JZ')
df['postcode_district'] = df['postcode'].str.split().str[0]
print(f'Unique postcode districts: {df["postcode_district"].nunique()}')
print('Top 15:')
print(df['postcode_district'].value_counts().head(15))

Unique postcode districts: 46
Top 15:
postcode_district
HP13    13293
HP22    10715
MK18    10406
HP21     9580
HP19     7812
HP5      7044
HP18     6965
HP12     6194
HP11     5988
SL9      5847
SL7      5725
HP10     5388
HP9      5027
HP14     4983
HP20     4953
Name: count, dtype: int64


### 3.7 Final engineered feature set

In [23]:
# Summarise what we've added
engineered = ['fabric_score', 'systems_score', 'building_age_years',
              'energy_per_m2', 'heating_cost_per_m2', 'log_floor_area',
              'built_form_simple', 'postcode_district'] +              [c + '_ord' for c in eff_cols]

print(f'Engineered {len(engineered)} new features.')
print()
print('Sample correlations with current_energy_efficiency:')
numeric_engineered = [c for c in engineered
                      if df[c].dtype != 'object']
corrs = df[numeric_engineered + ['current_energy_efficiency']].corr()['current_energy_efficiency']
print(corrs.drop('current_energy_efficiency').sort_values(ascending=False))

Engineered 15 new features.

Sample correlations with current_energy_efficiency:
fabric_score                0.722956
walls_energy_eff_ord        0.660218
windows_energy_eff_ord      0.559555
roof_energy_eff_ord         0.540848
systems_score               0.521435
mainheatc_energy_eff_ord    0.482928
hot_water_energy_eff_ord    0.442596
lighting_energy_eff_ord     0.337038
mainheat_energy_eff_ord     0.306158
log_floor_area             -0.082831
energy_per_m2              -0.317948
building_age_years         -0.601684
heating_cost_per_m2        -0.707592
Name: current_energy_efficiency, dtype: float64
